# Retail HOL — Dynamic Iceberg Tables (Snowflake-Managed Storage)

---

## The Scenario

Imagine you run a **12-SKU candle and fragrance brand** — products like *Frosted Juniper* and *Vanilla Cedarlight* — sold online and through **4 regional store networks**, fulfilled out of **5 warehouses** (LA, Houston, NYC, Chicago, SF).

Your analytics team has a problem:

- The nightly batch pipeline takes **4 hours** to rebuild all history every run
- Customer sentiment data is only available **next-day** — too late to catch viral complaints
- The data science team wants **Spark access** to Gold-layer tables without Snowflake credentials
- Storage is growing 15% monthly because nothing gets archived

**Your mission:** Replace the brittle batch pipeline with a modern, declarative open-lakehouse architecture that:
1. Refreshes incrementally (only process what changed)
2. Skips re-processing old data (Frozen Regions)
3. Stores everything in open Apache Iceberg format for multi-engine access
4. Enriches data with AI inline — no external ML pipeline needed

---

## Why Dynamic *Iceberg* Tables?

Regular Dynamic Tables store data in Snowflake's proprietary FDN format. **Dynamic Iceberg Tables** store output in **Apache Iceberg format** on Snowflake-managed storage — giving you the best of both worlds:

| Benefit | Description |
|---------|-------------|
| **Open Format** | Parquet data files + Iceberg metadata — no vendor lock-in |
| **Multi-Engine Access** | Spark, Trino, Flink, Dremio can read the Gold layer directly via Snowflake Horizon Catalog |
| **Same DT Features** | ADAPTIVE refresh, Frozen Regions, SLP all work identically |
| **Interoperability** | Bronze (Iceberg via OpenFlow) → Gold (Iceberg via DT) = full open lakehouse |

## Architecture: Full Open Lakehouse

```
┌─────────────────┐     ┌──────────────────────┐     ┌───────────────────────────────────┐
│  PostgreSQL     │────▶│  Iceberg Tables      │────▶│  Dynamic Iceberg Tables           │
│  (Source OLTP)  │ CDC │  (Bronze Layer)      │ DIT │  (Sliver/Gold — Open Format Analytics) 
│                 │     │  Snowflake-managed   │     │  ADAPTIVE + Frozen + SLP          │
└─────────────────┘     └──────────────────────┘     └───────────────────────────────────┘
     OpenFlow              Apache Iceberg                Apache Iceberg on Snowflake Managed Storage
     Real-time CDC         Snowflake reads/writes        Spark + Trino can read using Horizon Catalog
```

Deep dive: [What's New in Dynamic Tables — Faster & More Flexible](https://www.snowflake.com/en/blog/whats-new-dynamic-tables-faster-flexible/)

**Note:** If the link does not work, it is because notebook can not direct to internet. Please use the link in your browser  -  https://www.snowflake.com/en/blog/whats-new-dynamic-tables-faster-flexible/

## Coming Soon -  Sub-Minute Target Lag (Private Preview)

Today, the minimum TARGET_LAG for Dynamic Tables is 1 minute. For use cases requiring near-real-time freshness — such as live inventory dashboards or fraud detection — Snowflake is previewing sub-minute target lag, enabling refresh cycles in the range of seconds. This means your Dynamic Tables can stay within seconds of the base data, closing the gap between batch and streaming without changing your declarative SQL definition.

Once generally available, you'll be able to set values like TARGET_LAG = '15 seconds' — same syntax, same ADAPTIVE refresh, just faster. No Kafka, no micro-batch hacks, no infrastructure changes.

Deep dive: [Low Latency Dynamic Tables ](https://docs.snowflake.com/en/LIMITEDACCESS/low-latency-dynamic-tables)

---

## Setup: Connect to Bronze Layer

OpenFlow replicated our PostgreSQL tables into the `BRONZE` schema as Iceberg tables. The connector runs under a separate runtime role, so we need to grant our HOL role access to those tables before we can transform them.

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

-- Grant USAGE on the BRONZE schema so RETAIL_HOL_ROLE can resolve it
GRANT USAGE ON SCHEMA RETAIL_HOL_ICEBERG.BRONZE TO ROLE RETAIL_HOL_ROLE;

-- Grant SELECT on all existing tables in BRONZE
GRANT SELECT ON ALL TABLES IN SCHEMA RETAIL_HOL_ICEBERG.BRONZE TO ROLE RETAIL_HOL_ROLE;
GRANT SELECT ON ALL ICEBERG TABLES IN SCHEMA RETAIL_HOL_ICEBERG.BRONZE TO ROLE RETAIL_HOL_ROLE;

-- Grant SELECT on future tables (so new tables created by the connector are automatically accessible)
GRANT SELECT ON FUTURE TABLES IN SCHEMA RETAIL_HOL_ICEBERG.BRONZE TO ROLE RETAIL_HOL_ROLE;
GRANT SELECT ON FUTURE ICEBERG TABLES IN SCHEMA RETAIL_HOL_ICEBERG.BRONZE TO ROLE RETAIL_HOL_ROLE;

In [ ]:
%%sql -r dataframe_2
-- Switch to HOL role and warehouse for the rest of the notebook
USE ROLE RETAIL_HOL_ROLE;
USE WAREHOUSE RETAIL_HOL_WH;
USE DATABASE RETAIL_HOL_ICEBERG;

In [ ]:
%%sql -r dataframe_3
-- Create schemas for our Silver and Gold layers
CREATE SCHEMA IF NOT EXISTS SILVER_MANAGED_ICEBERG;
CREATE SCHEMA IF NOT EXISTS GOLD_MANAGED_ICEBERG;

---

## Explore: What's in our Bronze Layer?

Before transforming anything, let's look at what OpenFlow replicated from Snowflake PostgreSQL. This is the raw, un-transformed data — exactly as it exists in the source system.

In [ ]:
%%sql -r dataframe_4
-- Sales data: each row is one product on one day, with regional unit columns
-- Notice the wide format — UNITS_ONLINE, UNITS_NE, UNITS_NW, etc. are separate columns
-- We'll unpivot these into normalized rows in Silver
SELECT PRODUCT_NAME, TRANSACTION_DATE, UNITS_ONLINE, UNITS_NE, UNITS_NW, UNITS_SE, UNITS_SW
FROM BRONZE.SALES_TRANSACTIONS
ORDER BY TRANSACTION_DATE DESC
LIMIT 5;

In [ ]:
%%sql -r dataframe_5
-- Inventory data: notice the METADATA column is semi-structured JSON
-- and WAREHOUSE_LOCATION is a geospatial point
SELECT SKU_NAME, SNAPSHOT_DATE, QTY_ONLINE, METADATA, TRY_TO_GEOMETRY(WAREHOUSE_LOCATION)
FROM BRONZE.INVENTORY_FEED
LIMIT 3;

In [ ]:
%%sql -r dataframe_6
-- Product reviews: free-text bodies that we'll enrich with Cortex AI
SELECT PRODUCT_NAME, TITLE, BODY, STAR_RATING
FROM BRONZE.PRODUCT_REVIEWS
LIMIT 3;

**What we see:** The Bronze layer has clean structure but needs work:
- Sales data is in **wide format** (one column per region) — hard to aggregate by region
- Inventory has **nested JSON** in the METADATA column as ``VARIANT`` Data Type — can't query supplier or quality directly
- Reviews are **raw text** — no sentiment scores, no categorization

Let's fix all of this in the Silver layer.

---
## Step 1: Silver Layer — Cleanse, Normalize, Enrich

The Silver layer transforms raw Bronze data into a queryable, analytics-friendly shape. We'll create **Dynamic Iceberg Tables** — Notice how clean the DDL is: because CATALOG and EXTERNAL_VOLUME are set at the database `RETAIL_HOL_ICEBERG` level, we don't repeat them on each table. The DDL requires:

- No `EXTERNAL_VOLUME`
- No `CATALOG`
- No `BASE_LOCATION`

Snowflake manages everything while keeping data in open Iceberg format internally.

### What each Silver table does:

| Table | Transformation | Key Technique |
|-------|---------------|---------------|
| `SILVER_SALES` | Unpivot wide→tall (5 regions become rows) | UNION ALL normalization |
| `SILVER_INVENTORY` | Flatten JSON, compute stock health, extract geo | Semi-structured + CASE + Geospatial |
| `SILVER_PRODUCT_REVIEWS` | Add AI sentiment + auto-categorize | Cortex AI inline enrichment |

### 1a. Silver Sales — Unpivot Regional Columns

The Bronze table has one column per region (`UNITS_ONLINE`, `UNITS_NE`, etc.). This wide format makes it impossible to filter or group by region. We'll unpivot into one row per region.

**`TARGET_LAG = 'DOWNSTREAM'`** means this table only refreshes when a downstream table (Gold) actually needs fresh data — saving compute when no one is querying.

In [ ]:
%%sql -r dataframe_8
-- Silver: Sales Transactions — unpivot wide regional columns into normalized rows
-- TARGET_LAG = 'DOWNSTREAM': only refresh when a Gold table needs this data
-- clean DDL - CATALOG and EXTERNAL_VOLUME are set at the database RETAIL_HOL_ICEBERG

CREATE OR REPLACE DYNAMIC ICEBERG TABLE SILVER_MANAGED_ICEBERG.SILVER_SALES
  TARGET_LAG = 'DOWNSTREAM'
  WAREHOUSE = RETAIL_HOL_WH
  REFRESH_MODE = ADAPTIVE
AS
  SELECT ID AS TRANSACTION_ID, PRODUCT_NAME, TRANSACTION_DATE,
         'ONLINE' AS REGION, UNITS_ONLINE AS UNITS_SOLD,
         (UNITS_ONLINE + UNITS_NE + UNITS_NW + UNITS_SE + UNITS_SW) AS TOTAL_UNITS,
         CREATED_AT
  FROM BRONZE.SALES_TRANSACTIONS
  UNION ALL
  SELECT ID, PRODUCT_NAME, TRANSACTION_DATE,
         'NORTHEAST', UNITS_NE,
         (UNITS_ONLINE + UNITS_NE + UNITS_NW + UNITS_SE + UNITS_SW),
         CREATED_AT
  FROM BRONZE.SALES_TRANSACTIONS
  UNION ALL
  SELECT ID, PRODUCT_NAME, TRANSACTION_DATE,
         'NORTHWEST', UNITS_NW,
         (UNITS_ONLINE + UNITS_NE + UNITS_NW + UNITS_SE + UNITS_SW),
         CREATED_AT
  FROM BRONZE.SALES_TRANSACTIONS
  UNION ALL
  SELECT ID, PRODUCT_NAME, TRANSACTION_DATE,
         'SOUTHEAST', UNITS_SE,
         (UNITS_ONLINE + UNITS_NE + UNITS_NW + UNITS_SE + UNITS_SW),
         CREATED_AT
  FROM BRONZE.SALES_TRANSACTIONS
  UNION ALL
  SELECT ID, PRODUCT_NAME, TRANSACTION_DATE,
         'SOUTHWEST', UNITS_SW,
         (UNITS_ONLINE + UNITS_NE + UNITS_NW + UNITS_SE + UNITS_SW),
         CREATED_AT
  FROM BRONZE.SALES_TRANSACTIONS;

In [ ]:
%%sql -r dataframe_9
-- silver_sales_peek
-- See the result: each region is now its own row — easy to filter and aggregate
SELECT PRODUCT_NAME, TRANSACTION_DATE, REGION, UNITS_SOLD, TOTAL_UNITS
FROM SILVER_MANAGED_ICEBERG.SILVER_SALES
WHERE PRODUCT_NAME = 'Sea Salt Blossom'
ORDER BY TRANSACTION_DATE DESC
LIMIT 10;

### 1b. Silver Inventory — Flatten JSON + Stock Health Classification

#### Working with VARIANT: Flattening Semi-Structured Data in Iceberg  V3 Spec 

The `METADATA` column in the Bronze inventory feed is stored as Snowflake's **VARIANT** type — a flexible,
schema-on-read container that holds arbitrary nested JSON. 

In the Silver Dynamic Table below, we flatten it into strongly-typed scalar columns using dot-notation path extraction and `::` casting:

We'll extract these into proper columns, compute a `STOCK_HEALTH` classification, and parse geospatial warehouse coordinates.

**Iceberg V3 Spec**  

Deep dive: [Apache Iceberg™ tables: Support for Apache Iceberg™ v3](https://docs.snowflake.com/en/user-guide/tables-iceberg-v3-specification-support)

**Note:** If the link does not work, it is because notebook can not direct to internet. Please use the link in your browser  -  https://docs.snowflake.com/en/user-guide/tables-iceberg-v3-specification-support


In [ ]:
%%sql -r dataframe_10
-- silver_inventory
-- Silver: Inventory Feed — flatten JSON, classify stock health, extract geolocation
-- clean DDL - CATALOG and EXTERNAL_VOLUME are set at the database RETAIL_HOL_ICEBERG
CREATE OR REPLACE DYNAMIC ICEBERG TABLE SILVER_MANAGED_ICEBERG.SILVER_INVENTORY
  TARGET_LAG = '5 minutes'
  WAREHOUSE = RETAIL_HOL_WH
  REFRESH_MODE = ADAPTIVE
AS
  SELECT
    ID AS INVENTORY_ID,
    SKU_NAME,
    SNAPSHOT_DATE,
    QTY_ONLINE,
    QTY_NORTHEAST,
    QTY_SOUTHEAST,
    QTY_SOUTHWEST,
    QTY_NORTHWEST,
    (QTY_ONLINE + QTY_NORTHEAST + QTY_SOUTHEAST + QTY_SOUTHWEST + QTY_NORTHWEST) AS TOTAL_STOCK,
    METADATA:supplier::VARCHAR AS SUPPLIER,
    METADATA:quality_score::FLOAT AS QUALITY_SCORE,
    METADATA:storage_temp_f::INT AS STORAGE_TEMP_F,
    METADATA:batch_id::VARCHAR AS BATCH_ID,
    METADATA:last_audit::DATE AS LAST_AUDIT_DATE,
    CASE
      WHEN (QTY_ONLINE + QTY_NORTHEAST + QTY_SOUTHEAST + QTY_SOUTHWEST + QTY_NORTHWEST) < 50000 THEN 'CRITICAL'
      WHEN (QTY_ONLINE + QTY_NORTHEAST + QTY_SOUTHEAST + QTY_SOUTHWEST + QTY_NORTHWEST) < 100000 THEN 'LOW'
      WHEN (QTY_ONLINE + QTY_NORTHEAST + QTY_SOUTHEAST + QTY_SOUTHWEST + QTY_NORTHWEST) < 200000 THEN 'NORMAL'
      ELSE 'OVERSTOCKED'
    END AS STOCK_HEALTH,
    ST_X(TO_GEOMETRY(WAREHOUSE_LOCATION)) AS WAREHOUSE_LONGITUDE,
    ST_Y(TO_GEOMETRY(WAREHOUSE_LOCATION)) AS WAREHOUSE_LATITUDE,
    CREATED_AT
  FROM BRONZE.INVENTORY_FEED;

In [ ]:
%%sql -r dataframe_11
-- silver_inventory_peek
-- JSON is now flat columns, stock health is classified, geo is extracted
SELECT SKU_NAME, TOTAL_STOCK, STOCK_HEALTH, SUPPLIER, QUALITY_SCORE,
       WAREHOUSE_LATITUDE, WAREHOUSE_LONGITUDE
FROM SILVER_MANAGED_ICEBERG.SILVER_INVENTORY
ORDER BY TOTAL_STOCK ASC
LIMIT 5;

### 1c. Silver Product Reviews — Cortex AI Inline Enrichment

We're calling **Cortex AI functions directly inside the Dynamic Table definition**:

- `SNOWFLAKE.CORTEX.SENTIMENT(BODY)` — returns a score from -1 (very negative) to +1 (very positive)
- `SNOWFLAKE.CORTEX.CLASSIFY_TEXT(BODY, [...categories...])` — auto-classifies each review into a theme

**The magic:** Every time a new review arrives in Bronze, the Dynamic Table automatically refreshes and the AI functions run on the new rows. 

**NOTE:** No external ML pipeline, no batch jobs, no API keys, no model hosting. Just SQL.

In [ ]:
%%sql -r dataframe_12
-- silver_reviews
-- Silver: Product Reviews — Cortex AI sentiment scoring + theme classification
-- Every new review gets AI-enriched automatically on refresh
-- clean DDL - CATALOG and EXTERNAL_VOLUME are set at the database RETAIL_HOL_ICEBERG
CREATE OR REPLACE DYNAMIC ICEBERG TABLE SILVER_MANAGED_ICEBERG.SILVER_PRODUCT_REVIEWS
  TARGET_LAG = '10 minutes'
  WAREHOUSE = RETAIL_HOL_WH
  REFRESH_MODE = ADAPTIVE
AS
  SELECT
    ID AS REVIEW_ID,
    PRODUCT_ID,
    PRODUCT_NAME,
    TITLE AS REVIEW_TITLE,
    BODY AS REVIEW_BODY,
    STAR_RATING,
    AUTHOR,
    SUBMITTED_AT,
    IS_VERIFIED,
    HELPFUL_VOTES,
    MODERATION_STATUS,
    API_SOURCE AS CHANNEL,
    SNOWFLAKE.CORTEX.SENTIMENT(BODY) AS AI_SENTIMENT_SCORE,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
      BODY,
      ['product_quality', 'shipping_packaging', 'scent_fragrance', 'value_for_money', 'customer_service']
    ):label::VARCHAR AS AI_REVIEW_THEME,
    CREATED_AT
  FROM BRONZE.PRODUCT_REVIEWS;

In [ ]:
%%sql -r dataframe_13
-- silver_reviews_peek
-- See the AI enrichment in action: sentiment scores and auto-classified themes
SELECT PRODUCT_NAME, REVIEW_TITLE, STAR_RATING,
       ROUND(AI_SENTIMENT_SCORE, 3) AS SENTIMENT,
       AI_REVIEW_THEME
FROM SILVER_MANAGED_ICEBERG.SILVER_PRODUCT_REVIEWS
ORDER BY AI_SENTIMENT_SCORE ASC
LIMIT 8;

**Silver Layer Complete.** We went from messy, wide-format, JSON-laden raw data to:
- Normalized sales rows (filterable by region)
- Flat, classified inventory with geospatial coordinates
- AI-enriched reviews with sentiment scores and themes

All in open Iceberg format. All refreshing automatically. Now let's build the Gold layer that business users actually query.

---
## Step 2: Gold Layer — Business-Ready Analytics

The Gold layer joins Silver tables into **pre-aggregated, insight-ready** datasets. This is where we deploy two powerful cost-saving features:

### Frozen Regions

**The problem:** Without Frozen Regions, every refresh re-reads and re-computes ALL historical rows — even data from 6 months ago that will never change.

**The solution:** `FROZEN WHERE (date_col < CURRENT_DATE() - INTERVAL '7 days')` tells Snowflake: "rows matching this condition are immutable — skip them during refresh."

**The impact:** For a table with 6 months of history, only the last 7 days get refreshed — **cutting compute by ~95%**.

```
┌──────────────────────────────────────────────────────────────────┐
│                        DAILY_PRODUCT_PERFORMANCE                  │
├────────────────────────────────────────────┬─────────────────────┤
│          FROZEN (skip refresh)             │  LIVE (refreshed)   │
│  Jan ──── Feb ──── Mar ──── ... ──── -7d  │  -7d ──── today     │
│  ~95% of rows: ZERO compute cost           │  ~5% of rows only   │
└────────────────────────────────────────────┴─────────────────────┘
```

> Deep dive: [What's New in Dynamic Tables — Faster & More Flexible](https://www.snowflake.com/en/blog/whats-new-dynamic-tables-faster-flexible/)

### 2a. Daily Product Performance — The 360° View

This is the flagship Gold table. It joins **sales + inventory + reviews** into a single daily performance row per product. Your analytics team can answer questions like:
- "Which products are selling well but running low on stock?"
- "Is there a correlation between sentiment drops and sales declines?"

**Frozen Region:** Rows older than 7 days are frozen — only the most recent week refreshes.

In [ ]:
%%sql -r dataframe_14
-- gold_perf
-- Gold: Daily Product Performance — joins sales + inventory + sentiment
-- FROZEN WHERE: rows older than 7 days are immutable, skip refresh entirely
-- clean DDL - CATALOG and EXTERNAL_VOLUME are set at the database RETAIL_HOL_ICEBERG
CREATE OR REPLACE DYNAMIC ICEBERG TABLE GOLD_MANAGED_ICEBERG.DAILY_PRODUCT_PERFORMANCE
  TARGET_LAG = '10 minutes'
  WAREHOUSE = RETAIL_HOL_WH
  REFRESH_MODE = ADAPTIVE
  FROZEN WHERE (PERFORMANCE_DATE < CURRENT_DATE() - INTERVAL '7 days')
AS
  SELECT
    s.TRANSACTION_DATE AS PERFORMANCE_DATE,
    s.PRODUCT_NAME,
    SUM(s.TOTAL_UNITS) AS TOTAL_UNITS_SOLD,
    SUM(CASE WHEN s.REGION = 'ONLINE' THEN s.UNITS_SOLD ELSE 0 END) AS ONLINE_UNITS,
    SUM(CASE WHEN s.REGION != 'ONLINE' THEN s.UNITS_SOLD ELSE 0 END) AS STORE_UNITS,
    MAX(i.TOTAL_STOCK) AS INVENTORY_POSITION,
    MAX(i.STOCK_HEALTH) AS STOCK_HEALTH_STATUS,
    MAX(i.QUALITY_SCORE) AS QUALITY_SCORE,
    AVG(r.SENTIMENT_SUM) AS AVG_DAILY_SENTIMENT,
    SUM(r.POSITIVE_COUNT) AS POSITIVE_REVIEWS,
    SUM(r.NEGATIVE_COUNT) AS NEGATIVE_REVIEWS
  FROM SILVER_MANAGED_ICEBERG.SILVER_SALES s
  LEFT JOIN SILVER_MANAGED_ICEBERG.SILVER_INVENTORY i
    ON s.PRODUCT_NAME = i.SKU_NAME
    AND i.SNAPSHOT_DATE = DATE_TRUNC('WEEK', s.TRANSACTION_DATE)
  LEFT JOIN BRONZE.DAILY_REVIEWS r
    ON s.PRODUCT_NAME = r.PRODUCT
    AND s.TRANSACTION_DATE = r.REVIEW_DATE
  GROUP BY 1, 2;

### 2b. Customer Voice Analytics — AI-Powered Comment Analysis

Every customer comment gets automatically:
- **Sentiment scored** — how positive/negative is it?
- **Summarized** — Cortex generates a one-line summary
- **Intent classified** — is it praise, a complaint, a question, a suggestion, or a shipping issue?

**Frozen Region:** Comments older than 14 days are frozen — their AI analysis never needs to be recomputed.

In [ ]:
%%sql -r dataframe_15
-- gold_voice
-- Gold: Customer Voice Analytics — full Cortex AI enrichment pipeline
-- SENTIMENT + SUMMARIZE + CLASSIFY_TEXT — all inline, all automatic
-- Frozen region: comments older than 14 days never re-processed
-- clean DDL - CATALOG and EXTERNAL_VOLUME are set at the database RETAIL_HOL_ICEBERG

CREATE OR REPLACE DYNAMIC ICEBERG TABLE GOLD_MANAGED_ICEBERG.CUSTOMER_VOICE_ANALYTICS
  TARGET_LAG = '15 minutes'
  WAREHOUSE = RETAIL_HOL_WH
  REFRESH_MODE = ADAPTIVE
  FROZEN WHERE (COMMENT_DATE < CURRENT_DATE() - INTERVAL '14 days')
AS
  SELECT
    c.ID AS COMMENT_ID,
    c.ITEM_NAME AS PRODUCT_NAME,
    c.COMMENT_TITLE,
    c.COMMENT_BODY,
    c.USER_NAME,
    c.CREATED_AT::DATE AS COMMENT_DATE,
    c.CHANNEL,
    SNOWFLAKE.CORTEX.SENTIMENT(c.COMMENT_BODY) AS SENTIMENT_SCORE,
    SNOWFLAKE.CORTEX.SUMMARIZE(c.COMMENT_BODY) AS AI_SUMMARY,
    SNOWFLAKE.CORTEX.CLASSIFY_TEXT(
      c.COMMENT_BODY,
      ['praise', 'complaint', 'question', 'suggestion', 'shipping_issue']
    ):label::VARCHAR AS CUSTOMER_INTENT,
    c.CREATED_AT
  FROM BRONZE.CUSTOMER_COMMENTS c;

### 2c. Weekly Product Scorecard — Executive Rollup

This is the table e.g.  VP of Product opens every Monday morning. It rolls up daily performance into weekly metrics with channel mix, sentiment trends, and review ratios.

**Frozen Region:** Only the current month refreshes — historical weeks are locked. `TARGET_LAG = '1 hour'` because executives don't need sub-minute freshness.

**Note:** This table reads from `DAILY_PRODUCT_PERFORMANCE` (another Dynamic Table), creating a **chained pipeline**. Snowflake manages the dependency graph automatically.

In [ ]:
%%sql -r dataframe_16
-- gold_scorecard
-- Gold: Weekly Product Scorecard — executive rollup
-- Reads from another DT (DAILY_PRODUCT_PERFORMANCE) — chained pipeline!
-- Frozen 30 days = only refresh current month
-- clean DDL - CATALOG and EXTERNAL_VOLUME are set at the database RETAIL_HOL_ICEBERG
CREATE OR REPLACE DYNAMIC ICEBERG TABLE GOLD_MANAGED_ICEBERG.WEEKLY_PRODUCT_SCORECARD
  TARGET_LAG = '1 hour'
  WAREHOUSE = RETAIL_HOL_WH
  REFRESH_MODE = ADAPTIVE
  FROZEN WHERE (WEEK_START < CURRENT_DATE() - INTERVAL '30 days')
AS
  SELECT
    DATE_TRUNC('WEEK', PERFORMANCE_DATE)::DATE AS WEEK_START,
    PRODUCT_NAME,
    SUM(TOTAL_UNITS_SOLD) AS WEEKLY_UNITS_SOLD,
    SUM(ONLINE_UNITS) AS WEEKLY_ONLINE_UNITS,
    SUM(STORE_UNITS) AS WEEKLY_STORE_UNITS,
    ROUND(SUM(ONLINE_UNITS) * 100.0 / NULLIF(SUM(TOTAL_UNITS_SOLD), 0), 1) AS ONLINE_MIX_PCT,
    AVG(INVENTORY_POSITION) AS AVG_INVENTORY,
    MAX(STOCK_HEALTH_STATUS) AS WORST_STOCK_STATUS,
    AVG(QUALITY_SCORE) AS AVG_QUALITY_SCORE,
    AVG(AVG_DAILY_SENTIMENT) AS WEEKLY_SENTIMENT,
    SUM(POSITIVE_REVIEWS) AS WEEKLY_POSITIVE,
    SUM(NEGATIVE_REVIEWS) AS WEEKLY_NEGATIVE,
    ROUND(SUM(POSITIVE_REVIEWS) * 100.0 /
      NULLIF(SUM(POSITIVE_REVIEWS) + SUM(NEGATIVE_REVIEWS), 0), 1) AS POSITIVE_REVIEW_PCT
  FROM GOLD_MANAGED_ICEBERG.DAILY_PRODUCT_PERFORMANCE
  GROUP BY 1, 2;

---
## Step 3: Verify the Pipeline

Let's confirm everything is running: all tables should show as Iceberg, ADAPTIVE, with their respective Frozen Regions active.

In [ ]:
%%sql -r dataframe_18
-- silver_dts
-- Silver layer Dynamic Iceberg Tables
SHOW DYNAMIC TABLES IN SCHEMA SILVER_MANAGED_ICEBERG;

In [ ]:
%%sql -r dataframe_19
-- silver_details
SELECT
  "name" AS DT_NAME,
  "refresh_mode" AS REFRESH_MODE,
  "target_lag" AS TARGET_LAG,
  "is_iceberg" AS IS_ICEBERG,
  "scheduling_state" AS STATE
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

In [ ]:
%%sql -r dataframe_20
-- gold_dts
-- Gold layer Dynamic Iceberg Tables
SHOW DYNAMIC TABLES IN SCHEMA GOLD_MANAGED_ICEBERG;

In [ ]:
%%sql -r dataframe_21
-- gold_details
-- Verify: all are Iceberg, ADAPTIVE, with frozen regions
SELECT
  "name" AS DT_NAME,
  "refresh_mode" AS REFRESH_MODE,
  "target_lag" AS TARGET_LAG,
  "immutable_where" AS FROZEN_REGION,
  "is_iceberg" AS IS_ICEBERG,
  "scheduling_state" AS STATE
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Pipeline DAG

Here's the full dependency graph we just built — Snowflake manages refresh ordering automatically:

```
Bronze (Iceberg via OpenFlow — real-time CDC)
  │
  ├── SILVER_SALES (TARGET_LAG = DOWNSTREAM)
  │     └──┐
  │        ├── DAILY_PRODUCT_PERFORMANCE (10 min, frozen 7d)
  │        │     └── WEEKLY_PRODUCT_SCORECARD (1 hr, frozen 30d)
  │        │
  ├── SILVER_INVENTORY (5 min)
  │     └──┘
  │
  ├── SILVER_PRODUCT_REVIEWS (10 min)
  │
  └── CUSTOMER_VOICE_ANALYTICS (15 min, frozen 14d)
```

**Key observations:**
- `SILVER_SALES` only refreshes when `DAILY_PRODUCT_PERFORMANCE` needs it (DOWNSTREAM lag)
- `WEEKLY_PRODUCT_SCORECARD` chains off `DAILY_PRODUCT_PERFORMANCE` — a **DT-to-DT dependency**
- Each table has a different lag tuned to its business SLA

In [ ]:
%%sql -r dataframe_22
-- refresh_hist
-- Check recent refresh activity to confirm the pipeline is running
SELECT
  NAME,
  STATE,
  REFRESH_TRIGGER,
  REFRESH_START_TIME,
  REFRESH_END_TIME,
  DATEDIFF('second', REFRESH_START_TIME, REFRESH_END_TIME) AS DURATION_SEC
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY())
WHERE SCHEMA_NAME IN ('SILVER_MANAGED_ICEBERG', 'GOLD_MANAGED_ICEBERG')
ORDER BY REFRESH_START_TIME DESC
LIMIT 10;

---
## Step 4: Answer Business Questions

The pipeline is built and running. Let's use it to answer real business questions

This is to explore in SQL Way, in next Notebook "retail-hol-talk-to-data" we will explore this business Question via AI agent in SnowWork



In [ ]:
%%sql -r dataframe_23
-- biz_sentiment_risk
-- Q1: Which products have the WORST sentiment this week?
-- These are early warning signals — sentiment drops often precede sales declines
SELECT
  PRODUCT_NAME,
  WEEKLY_UNITS_SOLD,
  ROUND(WEEKLY_SENTIMENT, 3) AS SENTIMENT,
  POSITIVE_REVIEW_PCT,
  WORST_STOCK_STATUS
FROM GOLD_MANAGED_ICEBERG.WEEKLY_PRODUCT_SCORECARD
WHERE WEEK_START = (SELECT MAX(WEEK_START) FROM GOLD_MANAGED_ICEBERG.WEEKLY_PRODUCT_SCORECARD)
ORDER BY WEEKLY_SENTIMENT ASC
LIMIT 5;

In [ ]:
%%sql -r dataframe_24
-- biz_channel_mix
-- Q2: How is our online vs. in-store mix trending?
SELECT
  WEEK_START,
  SUM(WEEKLY_ONLINE_UNITS) AS TOTAL_ONLINE,
  SUM(WEEKLY_STORE_UNITS) AS TOTAL_STORE,
  ROUND(SUM(WEEKLY_ONLINE_UNITS) * 100.0 /
    NULLIF(SUM(WEEKLY_ONLINE_UNITS) + SUM(WEEKLY_STORE_UNITS), 0), 1) AS ONLINE_PCT
FROM GOLD_MANAGED_ICEBERG.WEEKLY_PRODUCT_SCORECARD
GROUP BY WEEK_START
ORDER BY WEEK_START DESC
LIMIT 8;

---
## Recap: What We Built

In 15 minutes, you built a **fully declarative, incrementally refreshing, AI-enriched open lakehouse pipeline**:

| What | How | Benefit |
|------|-----|---------|  
| 3 Silver + 3 Gold tables | Dynamic Iceberg Tables | Zero orchestration — Snowflake manages refresh order |
| Incremental processing | ADAPTIVE refresh mode | Only changed micro-partitions are recomputed |
| Skip old data | Frozen Regions (7d / 14d / 30d) | ~95% compute savings on historical data |
| AI enrichment | Cortex SENTIMENT + CLASSIFY_TEXT + SUMMARIZE | No external ML pipeline, no API keys, no model hosting |
| Open format | Apache Iceberg (Parquet + metadata) | Spark, Trino, Flink can read Gold via Horizon IRC |
| Multi-engine access | Snowflake-managed storage | No S3 buckets or IAM to manage |

### The Old Way vs. The New Way

| | Old Pipeline | What You Just Built |
|---|---|---|
| Refresh | Full rebuild nightly (4 hrs) | Incremental, minutes, automatic |
| AI enrichment | Separate Python batch job | Inline SQL — zero infra |
| Multi-engine | Not possible (proprietary format) | Spark/Trino via Horizon IRC |
| Orchestration | Airflow DAG + cron + manual deps | Snowflake manages the DAG |

---
## Next Steps

Your pipeline is live and producing insights. In the next notebook, we'll make this data **conversational**:

**Notebook 04** (`04-retail-hol-talk-to-data.ipynb`): **Talk to Your Data** — Create Semantic Views over these Gold tables, build a multi-tool Cortex Agent, and register it with Snowflake Intelligence so anyone on the team can ask questions in plain English:

> *"Which candle had the biggest sentiment drop this week?"*  
> *"Show me products at risk of stockout in the Northeast."*  
> *"What are our top customer complaints about Vanilla Cedarlight?"*

No SQL required — the agent queries your Gold layer for them.